In [ ]:
%matplotlib inline
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

# 1. Loading Data

In [ ]:
# ==================== CONFIGURATION ====================
# Specify the results folder name here
RESULTS_FOLDER = "/Users/fadelbatal/Desktop/D3B Work/LGG_Paper_03_04_2026/Clinico_ResNet_Model/outputs"  # Change this to your folder name

# Define paths
data_dir = os.path.join(RESULTS_FOLDER, "data")
plots_dir = os.path.join(RESULTS_FOLDER, "plots")

# Create plots directory if it doesn't exist
os.makedirs(plots_dir, exist_ok=True)

print(f"Loading data from: {data_dir}")
print(f"Saving plots to: {plots_dir}")

In [ ]:
# read in data from the specified results folder
risk_stratification = None
for file_name in [
    "discovery_results.pkl",
    "replicate_results.pkl",
]:
    file_path = os.path.join(data_dir, file_name)
    
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found!")
        continue
        
    with open(file_path, mode="rb") as file:
        df = pickle.load(file)
        df["Cohort"] = file_name.split("_")[0].title()
        if risk_stratification is None:
            risk_stratification = df
        else:
            risk_stratification = pd.concat(
                [
                    risk_stratification,
                    df,
                ],
                ignore_index=True,
            )

if risk_stratification is None:
    raise ValueError(f"No data files found in {data_dir}")

print(f"Loaded {len(risk_stratification)} records")


# 2. Data Visualization

In [ ]:
# instantiate the KaplanMeierFitter
kmf = KaplanMeierFitter()

# set the colors - UPDATED FOR STANDARD/HIGH
colors = {"High": "red", "Low": "blue"}

# iterate through the cohorts: "Discovery" and "Replicate"
for i, cohort in enumerate(risk_stratification["Cohort"].unique()):
    # filter data for the current iteration
    cohort_data = risk_stratification[risk_stratification["Cohort"] == cohort]

    # create a figure and add a subplot for the line chart
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # prepare a survival table
    cell_texts, row_labels, col_labels = (
        [],
        [],
        [
            i * 20
            for i in range(int(max(cohort_data["Progression Free Survival"]) / 20) + 1)
        ],
    )

    # iterate through risk groups: "High" and "Standard"
    for risk_group in colors.keys():
        # Filter data for the current iteration
        group_data = cohort_data[cohort_data["Risk Group"] == risk_group]
        
        if len(group_data) == 0:
            print(f"Warning: No data for {risk_group} risk group in {cohort} cohort")
            continue

        # fit the model
        kmf.fit(
            group_data["Progression Free Survival"],
            event_observed=group_data["Event"].astype(int),
            label=risk_group,
        )

        # Plot the KM curve
        kmf.plot(
            ax=ax1,
            color=colors[risk_group],
            ci_show=False,
            legend=True,
        )

        # manually add censor markers
        censored_data = group_data[group_data["Event"] == False]
        ax1.plot(
            censored_data["Progression Free Survival"],
            kmf.survival_function_at_times(
                censored_data["Progression Free Survival"]
            ).values,
            "+",
            color=colors[risk_group],
            markersize=6,
        )

        # complete the survival table
        kmf.event_table["event_interval"] = list(
            map(lambda x: int(x // 20 * 20), kmf.event_table.index)
        )
        risk_at_event = (
            kmf.event_table.groupby("event_interval")["at_risk"].max().to_dict()
        )
        cell_text = [f"{risk_at_event.get(i, 0)}" for i in col_labels]

        cell_texts.append(cell_text)
        row_labels.append(risk_group)

    # plot the line chart on the top subplot
    ax1.set_xlabel("Time (Months)", fontsize=12)
    ax1.set_ylabel("Progression-free survival probability", fontsize=12)
    ax1.set_title(f"Clinico-ResNet - {cohort} Cohort", fontweight="bold", fontsize=14)
    legend = ax1.legend(title="Risk Group", loc="lower left", frameon=True)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_edgecolor('black')
    ax1.grid(True, alpha=0.3)

    # add a subplot for the table below the line chart
    ax2 = fig.add_subplot(212, frameon=False)
    ax2.axis("off")

    # append the table to the plot
    table = ax2.table(
        cellText=cell_texts,
        cellLoc="center",
        rowLabels=row_labels,
        rowColours=list(colors.values()),
        colLabels=col_labels,
        loc="bottom",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)

    # Save the figure
    output_path = os.path.join(plots_dir, f"KM_Curves_{cohort}.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()
    plt.close()

# 3. Logrank Test

In [ ]:
print("\n" + "="*50)
print("LOGRANK TEST RESULTS")
print("="*50 + "\n")

# Save logrank test results to file
logrank_results = []

# iterate through the cohorts: "Discovery" and "Replicate"
for i, cohort in enumerate(risk_stratification["Cohort"].unique()):
    # filter data for the current iteration
    cohort_data = risk_stratification[risk_stratification["Cohort"] == cohort]

    # perform the multivariate log-rank test between the two groups
    result = multivariate_logrank_test(
        cohort_data["Progression Free Survival"],
        cohort_data["Risk Group"],
        event_observed=cohort_data["Event"],
    )
    
    print(f"\n{cohort} Cohort:")
    print("-" * 50)
    result.print_summary()
    
    # Save results to list
    logrank_results.append({
        "Cohort": cohort,
        "Test Statistic": result.test_statistic,
        "p-value": result.p_value,
        "degrees_of_freedom": result.degrees_of_freedom
    })

# Save logrank results to CSV
logrank_df = pd.DataFrame(logrank_results)
logrank_output_path = os.path.join(RESULTS_FOLDER, "metrics", "logrank_test_results.csv")
logrank_df.to_csv(logrank_output_path, index=False)
print(f"\n\nLogrank test results saved to: {logrank_output_path}")

print(f"\n✓ All KM curves saved to: {plots_dir}")

In [ ]:
%matplotlib inline
import pickle
import os
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

# 1. Loading Data
# ==================== CONFIGURATION ====================
RESULTS_FOLDER = "/Users/fadelbatal/Desktop/D3B Work/LGG_Paper_03_04_2026/Clinico_ResNet_Model/outputs"

data_dir = os.path.join(RESULTS_FOLDER, "data")
plots_dir = os.path.join(RESULTS_FOLDER, "plots")
os.makedirs(plots_dir, exist_ok=True)

print(f"Loading data from: {data_dir}")
print(f"Saving plots to: {plots_dir}")

risk_stratification = None
for file_name in ["discovery_results.pkl", "replicate_results.pkl"]:
    file_path = os.path.join(data_dir, file_name)
    if not os.path.exists(file_path):
        print(f"Warning: {file_path} not found!")
        continue
    with open(file_path, mode="rb") as file:
        df = pickle.load(file)
        df["Cohort"] = file_name.split("_")[0].title()
        risk_stratification = df if risk_stratification is None else pd.concat(
            [risk_stratification, df], ignore_index=True
        )

if risk_stratification is None:
    raise ValueError(f"No data files found in {data_dir}")

print(f"Loaded {len(risk_stratification)} records")

# 2. Data Visualization
kmf = KaplanMeierFitter()
colors = {"High": "red", "Low": "blue"}

for i, cohort in enumerate(risk_stratification["Cohort"].unique()):
    cohort_data = risk_stratification[risk_stratification["Cohort"] == cohort]
    fig, ax1 = plt.subplots(figsize=(10, 6))

    cell_texts, row_labels, col_labels = (
        [],
        [],
        [i * 20 for i in range(int(max(cohort_data["Progression Free Survival"]) / 20) + 1)],
    )

    for risk_group in colors.keys():
        group_data = cohort_data[cohort_data["Risk Group"] == risk_group]
        if len(group_data) == 0:
            print(f"Warning: No data for {risk_group} risk group in {cohort} cohort")
            continue

        kmf.fit(
            group_data["Progression Free Survival"],
            event_observed=group_data["Event"].astype(int),
            label=risk_group,
        )
        kmf.plot(ax=ax1, color=colors[risk_group], ci_show=False, legend=True)

        censored_data = group_data[group_data["Event"] == False]
        ax1.plot(
            censored_data["Progression Free Survival"],
            kmf.survival_function_at_times(censored_data["Progression Free Survival"]).values,
            "+", color=colors[risk_group], markersize=6,
        )

        kmf.event_table["event_interval"] = list(
            map(lambda x: int(x // 20 * 20), kmf.event_table.index)
        )
        risk_at_event = kmf.event_table.groupby("event_interval")["at_risk"].max().to_dict()
        cell_texts.append([f"{risk_at_event.get(i, 0)}" for i in col_labels])
        row_labels.append(risk_group)

    # --- Log-rank test & annotation ---
    result = multivariate_logrank_test(
        cohort_data["Progression Free Survival"],
        cohort_data["Risk Group"],
        event_observed=cohort_data["Event"],
    )
    p_value = result.p_value
    p_text = "p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"

    ax1.annotate(
        f"Log-rank {p_text}",
        xy=(0.97, 0.97),
        xycoords="axes fraction",
        ha="right", va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.8),
    )

    ax1.set_xlabel("Time (Months)", fontsize=12)
    ax1.set_ylabel("Progression-free survival probability", fontsize=12)
    ax1.set_title(f"Clinico-ResNet - {cohort} Cohort", fontweight="bold", fontsize=14)
    legend = ax1.legend(title="Risk Group", loc="lower left", frameon=True)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_edgecolor('black')
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(212, frameon=False)
    ax2.axis("off")
    table = ax2.table(
        cellText=cell_texts, cellLoc="center",
        rowLabels=row_labels, rowColours=list(colors.values()),
        colLabels=col_labels, loc="bottom",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)

    output_path = os.path.join(plots_dir, f"KM_Curves_{cohort}.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()
    plt.close()

# 3. Logrank Test (console + CSV)
print("\n" + "="*50)
print("LOGRANK TEST RESULTS")
print("="*50 + "\n")

logrank_results = []
for i, cohort in enumerate(risk_stratification["Cohort"].unique()):
    cohort_data = risk_stratification[risk_stratification["Cohort"] == cohort]
    result = multivariate_logrank_test(
        cohort_data["Progression Free Survival"],
        cohort_data["Risk Group"],
        event_observed=cohort_data["Event"],
    )
    print(f"\n{cohort} Cohort:")
    print("-" * 50)
    result.print_summary()
    logrank_results.append({
        "Cohort": cohort,
        "Test Statistic": result.test_statistic,
        "p-value": result.p_value,
        "degrees_of_freedom": result.degrees_of_freedom
    })

logrank_df = pd.DataFrame(logrank_results)
logrank_output_path = os.path.join(RESULTS_FOLDER, "metrics", "logrank_test_results.csv")
logrank_df.to_csv(logrank_output_path, index=False)
print(f"\nLogrank test results saved to: {logrank_output_path}")
print(f"\n✓ All KM curves saved to: {plots_dir}")